# AI21SemanticTextSplitter

本示例将介绍如何在 LangChain 中使用 AI21SemanticTextSplitter。

## 安装

In [ ]:
pip install langchain-ai21

## 环境设置

我们需要获取一个 AI21 API 密钥并将 `AI21_API_KEY` 设置为环境变量：

In [ ]:
import os
from getpass import getpass

if "AI21_API_KEY" not in os.environ:
    os.environ["AI21_API_KEY"] = getpass()

## 示例用法

This section provides examples of how to use the `useQuery` hook.

本节将提供 `useQuery` hook 的使用示例。

```jsx
import { useQuery } from '@tanstack/react-query';
import axios from 'axios';

function Todos() {
  const { isLoading, error, data } = useQuery({
    queryKey: ['todos'],
    queryFn: async () => {
      const { data } = await axios.get('https://jsonplaceholder.typicode.com/todos');
      return data;
    },
  });

  if (isLoading) return 'Loading...';

  if (error) return 'An error has occurred: ' + error.message;

  return (
    <ul>
      {data.map((todo) => (
        <li key={todo.id}>{todo.title}</li>
      ))}
    </ul>
  );
}
```

### Basic Usage

The `useQuery` hook accepts an object with at least a `queryKey` and a `queryFn`.

- `queryKey`: A unique key for the query. This key is used to identify the query and its data. It can be a string or an array of strings and other values.
- `queryFn`: An asynchronous function that fetches your data. It should return a promise that resolves with your data or throws an error.

### Basic Usage

`useQuery` hook 接受一个对象，该对象至少包含 `queryKey` 和 `queryFn`。

- `queryKey`: 查询的唯一标识符。此键用于标识查询及其数据。它可以是字符串，也可以是字符串和其他值的数组。
- `queryFn`: 一个用于获取数据的异步函数。它应该返回一个解析为你的数据或抛出错误的 Promise。

```jsx
import { useQuery } from '@tanstack/react-query';

function Example() {
  const { data } = useQuery({
    queryKey: ['todos'],
    queryFn: () => fetch('/api/todos').then((res) => res.json()),
  });

  return <div>{/* Render your data */}</div>;
}
```

### Query Options

You can pass additional options to `useQuery` to customize its behavior.

### 查询选项

你可以向 `useQuery` 传递额外的选项来定制其行为。

```jsx
import { useQuery } from '@tanstack/react-query';

function Example() {
  const { data, isLoading, isError, error } = useQuery({
    queryKey: ['todos'],
    queryFn: () => fetch('/api/todos').then((res) => res.json()),
    staleTime: 1000 * 60 * 5, // 5 minutes
    gcTime: 1000 * 60 * 60, // 1 hour
    refetchOnWindowFocus: false,
    retry: 3,
  });

  if (isLoading) return 'Loading...';
  if (isError) return 'An error has occurred: ' + error.message;

  return <div>{/* Render your data */}</div>;
}
```

### Dependent Queries

Dependent queries are queries that depend on the results of other queries. You can enable dependent queries by setting the `enabled` option to `false` or by passing a condition to it.

### 依赖查询

依赖查询是指依赖于其他查询结果的查询。你可以通过将 `enabled` 选项设置为 `false` 或传递一个条件来启用依赖查询。

```jsx
import { useQuery } from '@tanstack/react-query';

function UserProfile({ userId }) {
  // Fetch user data
  const { data: user, isLoading: isLoadingUser } = useQuery({
    queryKey: ['user', userId],
    queryFn: () => fetch(`/api/users/${userId}`).then((res) => res.json()),
  });

  // Fetch user's posts, but only if user data is available
  const { data: posts, isLoading: isLoadingPosts } = useQuery({
    queryKey: ['posts', user?.id], // Use user.id from the previous query
    queryFn: () => fetch(`/api/posts?userId=${user.id}`).then((res) => res.json()),
    enabled: !!user, // Only run this query if user data is available
  });

  if (isLoadingUser || isLoadingPosts) return 'Loading...';

  return (
    <div>
      <h1>{user.name}</h1>
      <ul>
        {posts.map((post) => (
          <li key={post.id}>{post.title}</li>
        ))}
      </ul>
    </div>
  );
}
```

### Paginated Queries

Paginated queries allow you to fetch data in pages. You can use the `useQuery` hook with a `queryKey` that includes a page number or identifier.

### 分页查询

分页查询允许你分批获取数据。你可以将 `useQuery` hook 与包含页码或标识符的 `queryKey` 一起使用。

```jsx
import { useQuery } from '@tanstack/react-query';
import { useState } from 'react';

function PaginatedTodos() {
  const [page, setPage] = useState(0);

  const { data, isLoading, isError, error } = useQuery({
    queryKey: ['todos', page], // Include the page number in the query key
    queryFn: () =>
      fetch(`/api/todos?page=${page}&limit=10`).then((res) => res.json()),
  });

  if (isLoading) return 'Loading...';
  if (isError) return 'An error has occurred: ' + error.message;

  return (
    <div>
      <ul>
        {data.map((todo) => (
          <li key={todo.id}>{todo.title}</li>
        ))}
      </ul>
      <button onClick={() => setPage((old) => Math.max(old - 1, 0))} disabled={page === 0}>
        Previous page
      </button>
      <button onClick={() => setPage((old) => old + 1)} disabled={!data || data.length < 10}>
        Next page
      </button>
    </div>
  );
}
```

### Infinite Queries

Infinite queries allow you to fetch data in an "infinite" scroll manner. You can use the `useInfiniteQuery` hook for this purpose.

### 无限查询

无限查询允许你以“无限”滚动的方式获取数据。你可以为此目的使用 `useInfiniteQuery` hook。

```jsx
import { useInfiniteQuery } from '@tanstack/react-query';
import axios from 'axios';

function InfiniteTodos() {
  const {
    data,
    fetchNextPage,
    hasNextPage,
    isFetchingNextPage,
    isLoading,
    isError,
    error,
  } = useInfiniteQuery({
    queryKey: ['todos'],
    queryFn: async ({ pageParam = 0 }) => {
      const { data } = await axios.get(`/api/todos?offset=${pageParam}&limit=10`);
      return data;
    },
    getNextPageParam: (lastPage, allPages) => {
      // Determine the next page's offset based on the last page's data
      // This logic will depend on your API's response structure
      return lastPage.nextOffset; // Assuming your API returns a 'nextOffset' property
    },
  });

  if (isLoading) return 'Loading...';
  if (isError) return 'An error has occurred: ' + error.message;

  return (
    <div>
      {data.pages.map((page, i) => (
        <React.Fragment key={i}>
          {page.map((todo) => (
            <p key={todo.id}>{todo.title}</p>
          ))}
        </React.Fragment>
      ))}
      <button
        onClick={() => fetchNextPage()}
        disabled={!hasNextPage || isFetchingNextPage}
      >
        {isFetchingNextPage
          ? 'Loading more...'
          : hasNextPage
          ? 'Load More'
          : 'Nothing more to load'}
      </button>
    </div>
  );
}

### 按语义拆分文本

此示例展示了如何使用 `AI21SemanticTextSplitter` 根据语义含义将文本分割成块。

In [ ]:
from langchain_ai21 import AI21SemanticTextSplitter

TEXT = (
    "We’ve all experienced reading long, tedious, and boring pieces of text - financial reports, "
    "legal documents, or terms and conditions (though, who actually reads those terms and conditions to be honest?).\n"
    "Imagine a company that employs hundreds of thousands of employees. In today's information "
    "overload age, nearly 30% of the workday is spent dealing with documents. There's no surprise "
    "here, given that some of these documents are long and convoluted on purpose (did you know that "
    "reading through all your privacy policies would take almost a quarter of a year?). Aside from "
    "inefficiency, workers may simply refrain from reading some documents (for example, Only 16% of "
    "Employees Read Their Employment Contracts Entirely Before Signing!).\nThis is where AI-driven summarization "
    "tools can be helpful: instead of reading entire documents, which is tedious and time-consuming, "
    "users can (ideally) quickly extract relevant information from a text. With large language models, "
    "the development of those tools is easier than ever, and you can offer your users a summary that is "
    "specifically tailored to their preferences.\nLarge language models naturally follow patterns in input "
    "(prompt), and provide coherent completion that follows the same patterns. For that, we want to feed "
    'them with several examples in the input ("few-shot prompt"), so they can follow through. '
    "The process of creating the correct prompt for your problem is called prompt engineering, "
    "and you can read more about it here."
)

semantic_text_splitter = AI21SemanticTextSplitter()
chunks = semantic_text_splitter.split_text(TEXT)

print(f"The text has been split into {len(chunks)} chunks.")
for chunk in chunks:
    print(chunk)
    print("====")

### 按语义合并拆分文本

此示例展示了如何使用 `AI21SemanticTextSplitter` 根据语义含义将文本分割成块，然后根据 `chunk_size` 合并这些块。

In [ ]:
from langchain_ai21 import AI21SemanticTextSplitter

TEXT = (
    "We’ve all experienced reading long, tedious, and boring pieces of text - financial reports, "
    "legal documents, or terms and conditions (though, who actually reads those terms and conditions to be honest?).\n"
    "Imagine a company that employs hundreds of thousands of employees. In today's information "
    "overload age, nearly 30% of the workday is spent dealing with documents. There's no surprise "
    "here, given that some of these documents are long and convoluted on purpose (did you know that "
    "reading through all your privacy policies would take almost a quarter of a year?). Aside from "
    "inefficiency, workers may simply refrain from reading some documents (for example, Only 16% of "
    "Employees Read Their Employment Contracts Entirely Before Signing!).\nThis is where AI-driven summarization "
    "tools can be helpful: instead of reading entire documents, which is tedious and time-consuming, "
    "users can (ideally) quickly extract relevant information from a text. With large language models, "
    "the development of those tools is easier than ever, and you can offer your users a summary that is "
    "specifically tailored to their preferences.\nLarge language models naturally follow patterns in input "
    "(prompt), and provide coherent completion that follows the same patterns. For that, we want to feed "
    'them with several examples in the input ("few-shot prompt"), so they can follow through. '
    "The process of creating the correct prompt for your problem is called prompt engineering, "
    "and you can read more about it here."
)

semantic_text_splitter_chunks = AI21SemanticTextSplitter(chunk_size=1000)
chunks = semantic_text_splitter_chunks.split_text(TEXT)

print(f"The text has been split into {len(chunks)} chunks.")
for chunk in chunks:
    print(chunk)
    print("====")

### 将文本拆分为文档

本示例展示了如何使用 `AI21SemanticTextSplitter` 根据语义含义将文本分割成 `Document`。元数据将包含每个文档的类型。

In [ ]:
from langchain_ai21 import AI21SemanticTextSplitter

TEXT = (
    "We’ve all experienced reading long, tedious, and boring pieces of text - financial reports, "
    "legal documents, or terms and conditions (though, who actually reads those terms and conditions to be honest?).\n"
    "Imagine a company that employs hundreds of thousands of employees. In today's information "
    "overload age, nearly 30% of the workday is spent dealing with documents. There's no surprise "
    "here, given that some of these documents are long and convoluted on purpose (did you know that "
    "reading through all your privacy policies would take almost a quarter of a year?). Aside from "
    "inefficiency, workers may simply refrain from reading some documents (for example, Only 16% of "
    "Employees Read Their Employment Contracts Entirely Before Signing!).\nThis is where AI-driven summarization "
    "tools can be helpful: instead of reading entire documents, which is tedious and time-consuming, "
    "users can (ideally) quickly extract relevant information from a text. With large language models, "
    "the development of those tools is easier than ever, and you can offer your users a summary that is "
    "specifically tailored to their preferences.\nLarge language models naturally follow patterns in input "
    "(prompt), and provide coherent completion that follows the same patterns. For that, we want to feed "
    'them with several examples in the input ("few-shot prompt"), so they can follow through. '
    "The process of creating the correct prompt for your problem is called prompt engineering, "
    "and you can read more about it here."
)

semantic_text_splitter = AI21SemanticTextSplitter()
documents = semantic_text_splitter.split_text_to_documents(TEXT)

print(f"The text has been split into {len(documents)} Documents.")
for doc in documents:
    print(f"type: {doc.metadata['source_type']}")
    print(f"text: {doc.page_content}")
    print("====")

### 使用元数据创建文档

You can create documents with metadata by passing a `metadata` object to the `createDocument` function.

您可以通过将 `metadata` 对象传递给 `createDocument` 函数来创建带有元数据的文档。

```jsx
import { createDocument } from '@contentful/rich-text-types';

const document = createDocument({
  nodeType: 'document',
  content: [],
  metadata: {
    tags: [
      {
        sys: {
          type: 'Link',
          linkType: 'Tag',
          id: 'myTagId',
        },
      },
    ],
  },
});
```

The `metadata` object can contain a `tags` array. Each tag in the array should be a Contentful Tag resource.

`metadata` 对象可以包含一个 `tags` 数组。数组中的每个标签都应该是 Contentful 的 Tag 资源。

```jsx
import { createDocument } from '@contentful/rich-text-types';

const document = createDocument({
  nodeType: 'document',
  content: [],
  metadata: {
    tags: [
      {
        sys: {
          type: 'Link',
          linkType: 'Tag',
          id: 'myTagId',
        },
      },
      {
        sys: {
          type: 'Link',
          linkType: 'Tag',
          id: 'anotherTagId',
        },
      },
    ],
  },
});

本示例展示了如何使用 AI21SemanticTextSplitter 从文本创建 Documents，并为每个 Document 添加自定义元数据。

In [ ]:
from langchain_ai21 import AI21SemanticTextSplitter

TEXT = (
    "We’ve all experienced reading long, tedious, and boring pieces of text - financial reports, "
    "legal documents, or terms and conditions (though, who actually reads those terms and conditions to be honest?).\n"
    "Imagine a company that employs hundreds of thousands of employees. In today's information "
    "overload age, nearly 30% of the workday is spent dealing with documents. There's no surprise "
    "here, given that some of these documents are long and convoluted on purpose (did you know that "
    "reading through all your privacy policies would take almost a quarter of a year?). Aside from "
    "inefficiency, workers may simply refrain from reading some documents (for example, Only 16% of "
    "Employees Read Their Employment Contracts Entirely Before Signing!).\nThis is where AI-driven summarization "
    "tools can be helpful: instead of reading entire documents, which is tedious and time-consuming, "
    "users can (ideally) quickly extract relevant information from a text. With large language models, "
    "the development of those tools is easier than ever, and you can offer your users a summary that is "
    "specifically tailored to their preferences.\nLarge language models naturally follow patterns in input "
    "(prompt), and provide coherent completion that follows the same patterns. For that, we want to feed "
    'them with several examples in the input ("few-shot prompt"), so they can follow through. '
    "The process of creating the correct prompt for your problem is called prompt engineering, "
    "and you can read more about it here."
)

semantic_text_splitter = AI21SemanticTextSplitter()
texts = [TEXT]
documents = semantic_text_splitter.create_documents(
    texts=texts, metadatas=[{"pikachu": "pika pika"}]
)

print(f"The text has been split into {len(documents)} Documents.")
for doc in documents:
    print(f"metadata: {doc.metadata}")
    print(f"text: {doc.page_content}")
    print("====")

### 按起始索引拆分文本到文档

本示例展示了如何使用 AI21SemanticTextSplitter 根据语义含义将文本分割成 Documents。元数据将包含每个文档的起始索引。
请注意，起始索引指示了分块的顺序，而不是每个分块的实际起始索引。

In [ ]:
from langchain_ai21 import AI21SemanticTextSplitter

TEXT = (
    "We’ve all experienced reading long, tedious, and boring pieces of text - financial reports, "
    "legal documents, or terms and conditions (though, who actually reads those terms and conditions to be honest?).\n"
    "Imagine a company that employs hundreds of thousands of employees. In today's information "
    "overload age, nearly 30% of the workday is spent dealing with documents. There's no surprise "
    "here, given that some of these documents are long and convoluted on purpose (did you know that "
    "reading through all your privacy policies would take almost a quarter of a year?). Aside from "
    "inefficiency, workers may simply refrain from reading some documents (for example, Only 16% of "
    "Employees Read Their Employment Contracts Entirely Before Signing!).\nThis is where AI-driven summarization "
    "tools can be helpful: instead of reading entire documents, which is tedious and time-consuming, "
    "users can (ideally) quickly extract relevant information from a text. With large language models, "
    "the development of those tools is easier than ever, and you can offer your users a summary that is "
    "specifically tailored to their preferences.\nLarge language models naturally follow patterns in input "
    "(prompt), and provide coherent completion that follows the same patterns. For that, we want to feed "
    'them with several examples in the input ("few-shot prompt"), so they can follow through. '
    "The process of creating the correct prompt for your problem is called prompt engineering, "
    "and you can read more about it here."
)

semantic_text_splitter = AI21SemanticTextSplitter(add_start_index=True)
documents = semantic_text_splitter.create_documents(texts=[TEXT])
print(f"The text has been split into {len(documents)} Documents.")
for doc in documents:
    print(f"start_index: {doc.metadata['start_index']}")
    print(f"text: {doc.page_content}")
    print("====")

### 分割文档

本示例展示了如何使用 AI21SemanticTextSplitter 根据语义含义将文档列表分割成块。

In [ ]:
from langchain_ai21 import AI21SemanticTextSplitter
from langchain_core.documents import Document

TEXT = (
    "We’ve all experienced reading long, tedious, and boring pieces of text - financial reports, "
    "legal documents, or terms and conditions (though, who actually reads those terms and conditions to be honest?).\n"
    "Imagine a company that employs hundreds of thousands of employees. In today's information "
    "overload age, nearly 30% of the workday is spent dealing with documents. There's no surprise "
    "here, given that some of these documents are long and convoluted on purpose (did you know that "
    "reading through all your privacy policies would take almost a quarter of a year?). Aside from "
    "inefficiency, workers may simply refrain from reading some documents (for example, Only 16% of "
    "Employees Read Their Employment Contracts Entirely Before Signing!).\nThis is where AI-driven summarization "
    "tools can be helpful: instead of reading entire documents, which is tedious and time-consuming, "
    "users can (ideally) quickly extract relevant information from a text. With large language models, "
    "the development of those tools is easier than ever, and you can offer your users a summary that is "
    "specifically tailored to their preferences.\nLarge language models naturally follow patterns in input "
    "(prompt), and provide coherent completion that follows the same patterns. For that, we want to feed "
    'them with several examples in the input ("few-shot prompt"), so they can follow through. '
    "The process of creating the correct prompt for your problem is called prompt engineering, "
    "and you can read more about it here."
)

semantic_text_splitter = AI21SemanticTextSplitter()
document = Document(page_content=TEXT, metadata={"hello": "goodbye"})
documents = semantic_text_splitter.split_documents([document])
print(f"The document list has been split into {len(documents)} Documents.")
for doc in documents:
    print(f"text: {doc.page_content}")
    print(f"metadata: {doc.metadata}")
    print("====")